# nb_setup_01_bootstrap — create config + delta tables

Idempotently provisions the OneLake data model for the S3 → AI Search ingestion solution:
- `config` (single source of truth for runtime settings)
- `file_metadata` (change detection + `process_status` state machine)
- `ingestion_state`, `ingestion_log`, `skipped_log`

Run once at setup, and any time you change schemas or config defaults.
See `PRODUCT_SPEC.md` sections 4–6.


## Parameters


In [ ]:
# Lakehouse-relative locations (attach this notebook to the target lakehouse).
ACLS_FILE_PATH = 'Files/acls/acls.json'   # uploaded ACL definition


## Config defaults
Auth is **hybrid** (a Fabric constraint, not a design choice):
- **Document Intelligence + Azure OpenAI** use keyless Entra tokens
  (`notebookutils.credentials.getToken('https://cognitiveservices.azure.com')`) because the
  subscription enforces `disableLocalAuth=true` on Cognitive Services.
- **Azure AI Search** uses an **admin API key stored in Key Vault**. Fabric cannot mint a token
  for the AI Search audience, and Search is *not* under the `disableLocalAuth` policy, so a key is
  used. Notebooks read it with `notebookutils.credentials.getSecret(<vault>, <secret>)`.

`kv_name` + `search_key_secret` locate that secret. No secret value is stored here or in the
`config` table — only the vault + secret *name*.


In [ ]:
from datetime import datetime, timezone

DEFAULT_CONFIG = {
    'acl_bypass_enabled': 'false',
    'acls_file_path': ACLS_FILE_PATH,
    'source_mode': 's3_direct',        # s3_direct (S3 REST + SigV4, no boto3) [default] | s3_shortcut (OneLake shortcut)
    'shortcut_root': 'Files/s3_mmx_bucket',
    's3_endpoint_url': 'https://s3.<your-region>.amazonaws.com',  # s3_direct: AWS or S3-compatible (Cohesity/MinIO)
    's3_bucket': '<your-bucket>',       # s3_direct: bucket name
    's3_prefix': '',                    # s3_direct: only list keys under this prefix ('' = whole bucket)
    's3_region': 'us-east-1',           # s3_direct: SigV4 signing region (must match the bucket's region)
    's3_addressing': 'path',            # s3_direct: path | virtual (host style)
    's3_verify_tls': 'true',            # s3_direct: 'false' only for self-signed test endpoints
    's3_access_key_secret': 's3-access-key',  # s3_direct: Key Vault secret NAME holding the S3 access key id
    's3_secret_key_secret': 's3-secret-key',  # s3_direct: Key Vault secret NAME holding the S3 secret access key
    'doc_intelligence_endpoint': 'https://<your-di>.cognitiveservices.azure.com/',
    'doc_intelligence_endpoints': '',  # comma-separated DI pool (round-robin); '' falls back to the singular endpoint
    'doc_intelligence_model': 'prebuilt-layout',
    'aoai_endpoint': 'https://<your-aoai>.openai.azure.com/',
    'aoai_endpoints': '',              # comma-separated AOAI pool (round-robin); '' falls back to the singular endpoint
    'aoai_embedding_deployment': 'text-embedding-3-large',
    'embedding_dimensions': '3072',
    'search_endpoint': 'https://<your-search>.search.windows.net',
    'search_index_name': 'docs-rag',
    'search_key_secret': 'search-admin-key',  # Key Vault secret NAME holding the Search admin key
    'chunk_size': '8000',
    'chunk_strategy_version': 'page-v1',
    'max_concurrency': '8',
    'max_retries': '3',
    'ingesting_lease_minutes': '120',
    'supported_extensions': 'pdf,docx,pptx,xlsx,html,htm,txt,md',
    'backfill_mode': 'false',
    'batch_size': '100',
    'max_batches_per_run': '0',
    'run_time_budget_min': '0',
    'backfill_batch_size': '500',
    'kv_name': '<your-keyvault-name>',
}


## Create delta tables (idempotent)


In [ ]:
spark.sql('''
CREATE TABLE IF NOT EXISTS file_metadata (
    file_path           STRING,
    file_name           STRING,
    file_extension      STRING,
    file_size           BIGINT,
    modified_datetime   TIMESTAMP,
    author              STRING,
    etag                STRING,
    content_hash        STRING,
    change_hash         STRING,
    acl_version         STRING,
    last_seen_utc       TIMESTAMP,
    process_status      STRING,
    status_reason       STRING,
    retry_count         INT,
    status_updated_utc  TIMESTAMP
) USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS ingestion_state (
    file_path              STRING,
    change_hash            STRING,
    acl_version            STRING,
    chunk_count            INT,
    embedding_model        STRING,
    chunk_strategy_version STRING,
    indexed_utc            TIMESTAMP
) USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS ingestion_log (
    file_path       STRING,
    chunks          INT,
    pages           INT,
    duration_ms     BIGINT,
    embedding_model STRING,
    di_model        STRING,
    run_id          STRING,
    ts_utc          TIMESTAMP
) USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS skipped_log (
    file_path STRING,
    reason    STRING,
    detail    STRING,
    run_id    STRING,
    ts_utc    TIMESTAMP
) USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS throttle_log (
    run_id  STRING,
    ts_utc  TIMESTAMP,
    host    STRING,
    method  STRING,
    status  INT,
    attempt INT,
    wait_s  DOUBLE,
    kind    STRING
) USING DELTA
''')
print('delta tables ready')


## Create + seed the `config` table (idempotent)
Existing keys are preserved; only missing keys get default values (so operator edits survive re-runs).


In [ ]:
from pyspark.sql import Row
from delta.tables import DeltaTable

spark.sql('''
CREATE TABLE IF NOT EXISTS config (
    key STRING, value STRING, value_type STRING, updated_utc TIMESTAMP
) USING DELTA
''')

now = datetime.now(timezone.utc)
rows = [Row(key=k, value=str(v), value_type='string', updated_utc=now) for k, v in DEFAULT_CONFIG.items()]
defaults_df = spark.createDataFrame(rows)

# MERGE: insert only keys that don't already exist (preserve operator overrides).
tgt = DeltaTable.forName(spark, 'config')
(tgt.alias('t')
   .merge(defaults_df.alias('s'), 't.key = s.key')
   .whenNotMatchedInsertAll()
   .execute())

print('config seeded:')
spark.table('config').orderBy('key').show(50, truncate=False)


## Helper: read config as a dict
Reused by the other notebooks.


In [ ]:
def load_config(spark):
    return {r['key']: r['value'] for r in spark.table('config').collect()}

cfg = load_config(spark)
print({k: cfg[k] for k in list(cfg)[:5]}, '...')


## Validate the ACL file is present (optional but recommended)


In [ ]:
import json
try:
    raw = spark.read.text(cfg.get('acls_file_path', ACLS_FILE_PATH), wholetext=True).collect()[0][0]
    acls = json.loads(raw)
    print('ACL folders defined:', len(acls.get('folders', [])))
except Exception as e:
    print('WARNING: ACL file not found or invalid ->', e)
    print('Upload config/acls.example.json to', cfg.get('acls_file_path', ACLS_FILE_PATH))
